# 狮头股份 600539.SH 均线交叉策略：信号生成、可视化与回测本 notebook 使用狮头股份前复权日线数据，演示完整的均线交叉策略流程：1. **加载已存储的股价数据** — 读取前复权 CSV 日线数据2. **计算均线数据** — 设定短均线 MA5 和长均线 MA153. **计算交易信号** — 金叉（MA5 上穿 MA15）买入，死叉（MA5 下穿 MA15）卖出4. **可视化图形** — 股价、长短均线、交易信号、买卖标记5. **策略回测** — 模拟交易并计算总收益率、年化收益、最大回撤、夏普比率、胜率等量化指标

## 1. 参数与依赖

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

STOCK_NAME = '狮头股份'
TS_CODE = '600539.SH'
DATA_PATH = Path('../data/shitou_600539_qfq_daily.csv')
META_PATH = Path('../data/shitou_600539_qfq_daily.json')
OUTPUT_DIR = Path('../output')
FIGURE_DIR = OUTPUT_DIR / 'figures'

SHORT_WINDOW = 5
LONG_WINDOW = 15
INITIAL_CAPITAL = 100000
COMMISSION_RATE = 0.0003
STAMP_TAX_RATE = 0.0005
SLIPPAGE = 0.001

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. 加载已存储的股价数据

In [ ]:
df = pd.read_csv(DATA_PATH, dtype={'date': str})
with META_PATH.open('r', encoding='utf-8') as f:
    metadata = json.load(f)['metadata']

df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df = df.sort_values('date').reset_index(drop=True)

print(f'股票: {STOCK_NAME} ({TS_CODE})')
print(f'数据范围: {df["date"].min().strftime("%Y-%m-%d")} ~ {df["date"].max().strftime("%Y-%m-%d")}')
print(f'交易日数: {len(df)} 天')
df[['date', 'open', 'high', 'low', 'close', 'volume_hands']].head(10)

## 3. 计算短均线和长均线设定短均线周期为 **5**（MA5），长均线周期为 **15**（MA15）。- **MA5（短均线）**：反映近 5 日收盘价均值，对价格变化敏感- **MA15（长均线）**：反映近 15 日收盘价均值，趋势更平滑

In [ ]:
df[f'ma_{SHORT_WINDOW}'] = df['close'].rolling(window=SHORT_WINDOW).mean()
df[f'ma_{LONG_WINDOW}'] = df['close'].rolling(window=LONG_WINDOW).mean()

df[['date', 'close', f'ma_{SHORT_WINDOW}', f'ma_{LONG_WINDOW}']].tail(15)

## 4. 计算买入卖出交易信号均线交叉策略的核心逻辑：- **金叉（买入信号）**：MA5 从下方穿越到 MA15 上方 → 买入- **死叉（卖出信号）**：MA5 从上方穿越到 MA15 下方 → 卖出

In [ ]:
df['ma_diff'] = df[f'ma_{SHORT_WINDOW}'] - df[f'ma_{LONG_WINDOW}']
df['ma_diff_prev'] = df['ma_diff'].shift(1)

df['signal'] = 0
golden_cross = (df['ma_diff_prev'] < 0) & (df['ma_diff'] > 0)
death_cross = (df['ma_diff_prev'] > 0) & (df['ma_diff'] < 0)
df.loc[golden_cross, 'signal'] = 1
df.loc[death_cross, 'signal'] = -1

df['position'] = 0
current_pos = 0
for i in range(len(df)):
    if df.loc[i, 'signal'] == 1:
        current_pos = 1
    elif df.loc[i, 'signal'] == -1:
        current_pos = 0
    df.loc[i, 'position'] = current_pos

buy_signals = df[df['signal'] == 1]
sell_signals = df[df['signal'] == -1]

print(f'买入信号（金叉）: {len(buy_signals)} 次')
for _, row in buy_signals.iterrows():
    print(f'  {row["date"].strftime("%Y-%m-%d")}  收盘价: {row["close"]:.2f}')
print(f'\n卖出信号（死叉）: {len(sell_signals)} 次')
for _, row in sell_signals.iterrows():
    print(f'  {row["date"].strftime("%Y-%m-%d")}  收盘价: {row["close"]:.2f}')

## 5. 可视化：股价 + 均线 + 交易信号- 股价收盘价曲线（蓝色）- MA5 短均线（橙色）、MA15 长均线（紫色）- 买入信号（红色 ↑）、卖出信号（绿色 ↓）

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(df['date'], df['close'], label='收盘价', color='#2563eb', linewidth=1.5)
ax.plot(df['date'], df[f'ma_{SHORT_WINDOW}'], label=f'MA{SHORT_WINDOW}', color='#f97316', linewidth=1.2)
ax.plot(df['date'], df[f'ma_{LONG_WINDOW}'], label=f'MA{LONG_WINDOW}', color='#7c3aed', linewidth=1.2)
ax.scatter(buy_signals['date'], buy_signals['close'] * 0.985, marker='^', color='#dc2626', s=150, zorder=5, label='买入')
ax.scatter(sell_signals['date'], sell_signals['close'] * 1.015, marker='v', color='#16a34a', s=150, zorder=5, label='卖出')
ax.set_title(f'{STOCK_NAME} 股价、均线与交易信号', fontsize=14, fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
plt.show()

## 6. 模拟交易回测与量化指标回测规则：信号日次日开盘价成交，考虑佣金（万三）、印花税（千0.5）、滑点（0.1%），满仓买入/清仓卖出。

In [ ]:
trades = []
cash = INITIAL_CAPITAL
shares = 0
for i in range(len(df) - 1):
    today = df.iloc[i]
    tomorrow = df.iloc[i + 1]
    trade_price = tomorrow['open']
    buy_price = trade_price * (1 + SLIPPAGE)
    sell_price = trade_price * (1 - SLIPPAGE)
    if today['signal'] == 1 and shares == 0:
        buy_shares = int(cash / (buy_price * 100)) * 100
        if buy_shares > 0:
            cost = buy_shares * buy_price
            commission = max(cost * COMMISSION_RATE, 5)
            cash -= (cost + commission)
            shares = buy_shares
            trades.append({'date': tomorrow['date'], 'action': 'BUY', 'price': buy_price, 'shares': buy_shares, 'amount': cost, 'commission': commission})
    elif today['signal'] == -1 and shares > 0:
        revenue = shares * sell_price
        commission = max(revenue * COMMISSION_RATE, 5)
        stamp_tax = revenue * STAMP_TAX_RATE
        cash += (revenue - commission - stamp_tax)
        trades.append({'date': tomorrow['date'], 'action': 'SELL', 'price': sell_price, 'shares': shares, 'amount': revenue, 'commission': commission, 'stamp_tax': stamp_tax})
        shares = 0

final_close = df.iloc[-1]['close']
final_value = cash + shares * final_close
print(f'初始资金: ¥{INITIAL_CAPITAL:,.0f}')
print(f'期末总资产: ¥{final_value:,.2f}')
print(f'净利润: ¥{final_value - INITIAL_CAPITAL:,.2f}')
print(f'交易次数: {len(trades)} 次')
pd.DataFrame(trades)

## 7. 量化指标汇总

In [ ]:
df['strategy_return'] = df['position'].shift(1) * df['close'].pct_change()
df.loc[df.index[0], 'strategy_return'] = 0
df['strategy_nav'] = INITIAL_CAPITAL * (1 + df['strategy_return']).cumprod()
df['benchmark_nav'] = INITIAL_CAPITAL * (1 + df['close'].pct_change().fillna(0)).cumprod()

total_return = (final_value - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
bench_return = (df.iloc[-1]['close'] - df.iloc[0]['close']) / df.iloc[0]['close'] * 100
ann_return = ((1 + total_return/100) ** (252/len(df)) - 1) * 100
df['drawdown'] = (df['strategy_nav'] - df['strategy_nav'].cummax()) / df['strategy_nav'].cummax()
max_dd = df['drawdown'].min() * 100
rf = 0.02/252
sharpe = np.sqrt(252) * (df['strategy_return'] - rf).mean() / (df['strategy_return'] - rf).std()

pnl_list = []
for i in range(0, len(trades), 2):
    if i+1 < len(trades):
        pnl = trades[i+1]['amount'] - trades[i]['amount'] - trades[i+1]['commission'] - trades[i+1].get('stamp_tax', 0) - trades[i]['commission']
        pnl_list.append(pnl)
wins = sum(1 for p in pnl_list if p > 0)
win_rate = wins / len(pnl_list) * 100 if pnl_list else 0

print(f'策略总收益率:   {total_return:.2f}%')
print(f'基准总收益率:   {bench_return:.2f}%')
print(f'超额收益:       {total_return - bench_return:.2f}%')
print(f'策略年化收益:   {ann_return:.2f}%')
print(f'最大回撤:       {max_dd:.2f}%')
print(f'夏普比率:       {sharpe:.2f}')
print(f'胜率:           {win_rate:.1f}% ({wins}/{len(pnl_list)})')

## 8. 策略净值 vs 买入持有基准

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(df['date'], df['strategy_nav'], label='均线策略净值', color='#dc2626', linewidth=2)
ax.plot(df['date'], df['benchmark_nav'], label='买入持有基准', color='#2563eb', linewidth=2, alpha=0.7, linestyle='--')
ax.axhline(INITIAL_CAPITAL, color='#64748b', linestyle=':', linewidth=1, label='初始资金')
ax.set_title(f'{STOCK_NAME} 策略净值 vs 买入持有基准', fontsize=14, fontweight='bold')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
plt.show()

## 9. 小结| 指标 | 说明 || --- | --- || 总收益率 | 策略回测期间累计收益 || 年化收益 | 折算为年化收益率 || 最大回撤 | 策略从最高点的最大跌幅 || 夏普比率 | 风险调整后收益，越高越好 || 胜率 | 盈利交易次数 / 总交易次数 |MA5/MA15 均线交叉策略在趋势明确的行情中表现较好，但在震荡市中容易产生频繁的虚假信号。建议结合其他指标（如成交量、RSI、MACD）来过滤虚假信号，提高策略胜率。